# 5.4 Sparse Surveys

Sparse surveys define which source/receiver/component combinations are written instead of recording every dense all-to-all trace. This is essential when acquisition geometry is large or when only a useful offset range should be retained. This tutorial builds both an offset-domain survey and an explicit product survey, then runs the offset-domain case and inspects the sparse trace result.

By the end, you should be able to replace dense all-to-all recording with sparse source-receiver selections and verify the written trace table before plotting.


## How To Read This Tutorial

Sparse surveys are how large acquisition designs stay tractable. Dense all-to-all recording is convenient for small examples, but realistic source/receiver/component products can be enormous. A sparse survey makes selection explicit and stores only the traces that matter.

This notebook compares two selection styles: geometric offset-domain selection and explicit source/receiver products. Both are survey definitions, not post-processing filters.

## Design Notes

Dense acquisition is compact to author but scales as `sources x receivers x components`. Sparse acquisition moves selection into a named `SparseSurvey`, which defines which source/receiver/component combinations are active.

| Sparse survey type | API pattern | Best use |
| --- | --- | --- |
| Offset domain | `SparseSurvey.offset_domain(...)` | Keep traces inside a geometric offset/azimuth range computed from source and receiver coordinates. |
| Explicit product | `SparseSurvey.from_product(...)` | Keep all combinations from selected source ids, receiver ids, and components. |
| Explicit pairs | `SparseSurvey.from_pairs(...)` | Reproduce a predesigned acquisition table. |
| Imported layout | HDF5/SPS survey helpers | Reuse an external sparse survey definition. |

The public sparse input fields are trace identity fields: source id, receiver id, component, and weight. Internal point ranges and sample maps are runtime details, so they should not be authored directly in tutorials.


## Imports

The examples use the public `import frequensolve as fs` API plus standard scientific Python tools for inspection and plotting. Keeping imports ordinary makes the notebook easier to reuse in analysis or operations notebooks.

In [ ]:

import numpy as np
import frequensolve as fs

u = fs.ureg


## Model And Source/Receiver Geometry

The same geometry is used for the dense count estimate and the sparse run. The sparse receiver group points to a named survey, so the receiver coordinates remain normal physical coordinates.


In [ ]:
project = fs.Project(
    name="project",
    pretty_name="sparse_surveys",
    path="./scratch/tutorials/sparse_surveys",
    log_level="INFO",
    log_to_console=True,
)

sim = project.new_simulation(
    name="sparse_surveys",
    physics="acoustic",
    dimension=2,
    units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
)

model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.0])
model.add_surface(name="top", depth=0.0 * u.km)
model.add_layer(name="water", properties={"Vp": 1.5 * u.km / u.s, "Rho": 1.0 * u.g / u.cm**3})
model.add_surface(name="bottom", depth=0.45 * u.km)
sim += model

sim += model.hex_mesh_generator([8, 4])
sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=5.0, f_high=30.0)
sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
sim += fs.BoundaryCondition(conditions=["pml"], boundaries=["x_min", "x_max", "z_max"], pml_wavelengths=0.75)

sources = [[0.2, 0.05], [0.5, 0.05], [0.8, 0.05]]
receivers = [[x, 0.04] for x in np.linspace(0.05, 0.95, 91)]
node = fs.ReceiverNode(name="hydrophone")
node.add_component(name="p", field="pressure")


## Offset-Domain Sparse Survey

This survey keeps traces whose absolute horizontal offset is between 0.1 km and 0.35 km. The dense count is shown for comparison; the exact sparse count is determined by the solver from source and receiver coordinates.

Offset-domain selection is useful when the physically meaningful part of the acquisition is geometric rather than a fixed id list. The notebook does not author offsets directly. It authors physical coordinates and offset bounds, then lets the runtime compute which pairs belong to the sparse trace layout.


In [ ]:
offset_survey = fs.SparseSurvey.offset_domain(
    "middle_offsets",
    min=0.1 * u.km,
    max=0.35 * u.km,
    metric="horizontal",
)

acq = fs.Acquisition(max_batch=3)
acq.add_source_group(kind="scalar", coords=sources)
acq.add_sparse_receiver_group("middle_offsets", node, coords=receivers, survey=offset_survey)
sim += acq

{
    "dense_trace_count": len(sources) * len(receivers),
    "survey_kind": offset_survey.kind,
    "offset_domain": offset_survey.offset_domain,
}


## Explicit Product Survey

When source ids and receiver ids are already known, an explicit survey can be authored directly. This cell does not replace the offset-domain survey above; it shows the alternate payload shape for selected ids.


In [ ]:
explicit = fs.SparseSurvey.from_product(
    "selected_pairs",
    sources=[1, 2],
    receivers=[10, 20, 30, 40],
    components="p",
)
explicit.to_fs(component_map={"p": 1})


## Run The Sparse Survey

The receiver group is sparse, so the trace output should contain only the selected source/receiver/component combinations. Inspect `traces.summary` and `survey_tables()` before plotting; those tables are the best way to confirm a sparse layout.


In [ ]:
sim += fs.Discretization()
sim += fs.SolverConfig(tolerance=1.0e-4, grids=3)

site = fs.LocalSite(shutdown_on_completion=True, verbose=True)
job = fs.TimeDomainJob(
    name="time_sparse_offsets",
    simulation=sim,
    f_min=0.0,
    f_max=30.0,
    T_max=0.9,
)
result = site.submit(job).wait()
traces = result.traces(upscale=4)
traces.summary


## Inspect Sparse Tables And Plot Traces

Sparse trace stores keep survey metadata alongside the data. Inspect `survey_tables()` before plotting so you know which source ids, receiver ids, components, and weights were written. That table is the sparse equivalent of checking dense receiver-group shape.

The gather plot then shows only the selected receiver set for one source. If the sparse gather looks unexpectedly empty or discontinuous, the first things to inspect are the offset bounds, coordinate units, and source/receiver positions used to build the survey.


In [ ]:
tables = traces.survey_tables()
tables.keys()


In [ ]:
wavelet = fs.RickerWavelet(f=12.0)
group = "middle_offsets"
component = "p"
source = traces.sources(group)[0]
gather = traces.td(group, component, source, wavelet, upscale=4, T_max=0.9)
fig, ax = fs.plot_gather(
    gather,
    A=2.0 * np.nanstd(np.real(gather.values)),
    cmap="gray",
    figsize=(9, 4),
    title=f"Sparse offset-domain gather, source {source}",
)


## Before Moving On

The sparse table is the truth. Before plotting, inspect which source ids, receiver ids, components, and weights were written. A sparse gather may look empty or discontinuous if the selection is too restrictive, and the table is the fastest way to diagnose that.

Use sparse surveys when acquisition design is part of the model. Use dense surveys only when the all-to-all product is genuinely small or pedagogically useful.

## Result Review Checklist

Sparse surveys are primarily about selection. The best review artifact is the sparse survey table, because it tells you which logical traces were written before plotting hides that structure.

| Artifact | What to confirm |
| --- | --- |
| Dense count estimate | The all-to-all count is large enough that sparse selection is meaningful. |
| Offset-domain payload | Bounds and metric match the intended source-receiver geometry. |
| `survey_tables()` | Source ids, receiver ids, components, and weights match the selected layout. |
| Sparse gather | Only the selected receiver/source combinations appear in the plotted data. |

If too few traces are written, check offset units and the metric first. If too many are written, narrow the offset bounds or switch to explicit products/pairs for exact control.
